# Translation Task

[video](https://www.youtube.com/watch?v=ISNdQcPhsts)

In [1]:
from pathlib import Path

from dotenv import load_dotenv
from torch.utils.tensorboard import SummaryWriter

import models.deep_learning.architectures.transformer.tasks.translation as trn

load_dotenv()
device = trn.get_device()

## Config

In [2]:
CONFIG = trn.Config(
    batch_size=8,
    num_epochs=50,
    lr=1e-4,
    src_seq_len=350,
    tgt_seq_len=350,
    d_model=512,
    num_layers=6,
    n_heads=8,
    forward_dim=2048,
    droptout=0.1,
    datasource="Helsinki-NLP/opus_books",
    src_lang="en",
    tgt_lang="es",
    model_basename="tmodel_",
)


In [3]:
writer = SummaryWriter(CONFIG.experiment_name)
Path(CONFIG.model_folder).mkdir(parents=True, exist_ok=True)

## Load Dataset (from HuggingFace)

In [4]:
raw_ds = trn.TranslationHFDataset.load_dataset(
    path=CONFIG.datasource,
    name=f"{CONFIG.src_lang}-{CONFIG.tgt_lang}",
    split="train",
)

## Tokenization

In [5]:
tokenizer_src = trn.get_or_build_tokenizer(
    Path(CONFIG.tokenizer_src_file), raw_ds, CONFIG.src_lang
)
tokenizer_tgt = trn.get_or_build_tokenizer(
    Path(CONFIG.tokenizer_tgt_file), raw_ds, CONFIG.tgt_lang
)

## Create dataloaders

In [6]:
train_dataloader, val_dataloader = trn.create_dataloaders(
    raw_ds, tokenizer_src, tokenizer_tgt, CONFIG
)

Original dataset size: 93470
Filtered dataset size: 93464
Removed: 6 samples (0.01%)


## Create model

In [7]:
model = trn.Translator(
    src_vocab_size=tokenizer_src.get_vocab_size(),
    tgt_vocab_size=tokenizer_tgt.get_vocab_size(),
    src_max_length=CONFIG.src_seq_len,
    tgt_max_length=CONFIG.tgt_seq_len,
    embed_size=CONFIG.d_model,
    heads=CONFIG.n_heads,
    num_layers=CONFIG.num_layers,
    forward_dimension=CONFIG.forward_dim,
    dropout=CONFIG.droptout,
).to(device)

## Train the model

In [8]:
trn.train(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    tokenizer_src=tokenizer_src,
    tokenizer_tgt=tokenizer_tgt,
    device=device,
    config=CONFIG,
    writer=writer,
)

No model to preload, starting from scratch


Processing Epoch 00:  19%|█▉        | 1992/10515 [07:01<30:01,  4.73it/s, loss=6.009]  


RuntimeError: MPS backend out of memory (MPS allocated: 10.53 GiB, other allocations: 12.02 GiB, max allowed: 22.64 GiB). Tried to allocate 108.95 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).